In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.1-flash-lite")

In [37]:
query = "현재 부산시 지방선거에서 부산시장 후보는 총 몇명이고, 후보의 이름은 누구야?"

In [38]:
from langchain.messages import HumanMessage

model.invoke([HumanMessage(content=query)])

AIMessage(content=[{'type': 'text', 'text': "현재 시점(2024년 5월)을 기준으로 말씀드리면, **현재 진행 중인 부산시장 선거는 없습니다.**\n\n부산광역시장 선거는 지방선거 일정에 따라 4년마다 치러지며, 가장 최근에 있었던 선거는 **2022년 6월 1일 제8회 전국동시지방선거**였습니다.\n\n참고를 위해 **지난 2022년 제8회 지방선거 당시 부산시장 후보**는 총 2명이었으며, 명단은 다음과 같습니다.\n\n1. **박형준** (국민의힘) - 당선\n2. **변성완** (더불어민주당)\n\n다음 부산광역시장 선거는 **2026년 6월 3일**에 예정된 '제9회 전국동시지방선거'입니다. 따라서 현재 시점에는 등록된 후보가 없습니다. 선거가 다가오면 선거관리위원회(중앙선관위 홈페이지)를 통해 후보자 정보를 확인하실 수 있습니다.", 'extras': {'signature': 'EjQKMgEMOdbHII7FkW07dqytndw90bo5/HYQPuB54VkSNGW8RBy17VmqXYS2PV2pePZDYAiP'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e77a4-2f1c-7c00-ad4b-4d1982b49cd9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 234, 'total_tokens': 258, 'input_token_details': {'cache_read': 0}})

---

In [39]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(results_separator=";\n")
result = search.invoke(query)

In [40]:
print(result)

snippet: 2 weeks ago - 국민의힘이 부산시장 후보 신청을 받은 결과, 박형준 현 시장과 주진우 의원 등 2명이 최종 도전장을 냈습니다. 둘의 치열한 경선이 예상되는 가운데, 더불어민주당 유력 후보인 전재수 의원 역시 이번주 후보 등록 서류를 ..., title: 부산시장 선거 여야 후보 윤곽 ′뚜렷, link: https://busanmbc.co.kr/01_new/new01_view.asp?idx=283353;
snippet: 3 days ago - 더불어민주당 전재수 후보 48.0%, 국민의힘 박형준 후보 39.0%, 개혁신당 정이한 후보 3.2%. 전 후보가 박 후보를 오차범위 밖에서 9% 포인트 앞섰습니다. [전재수 / 더불어민주당 부산시장 후보(지난 21일, 선거출정식)] "여러분들과 ..., title: 부산시장 여론조사 전재수 48.0% 박형준 39.0% 2026 ..., link: https://busanmbc.co.kr/01_new/new01_view.asp?idx=285700;
snippet: 2 weeks ago - 부산 성인 1006명.무선 ARS)에선 국민의힘 박형준(58.2%) 후보가 민주당 변성완(29.1%) 후보를 29.1%P 앞서고 있었고, 결국 박 후보는 역대 부산시장 최다 득표율(66.4%)로 당선됐다., title: 역대 지선과 다른 흐름 보이는 부산시장 선거...최종 결과는 어떻게? - 부산일보, link: https://mobile.busan.com/view/busan/view.php?code=2026051917081064316;
snippet: 2 days ago - [앵커] 3선에 도전하는 국민의힘 박형준 후보의 지지세도 만만치 않죠? [기자] 그렇습니다. 보수민심이 예전 같지 않다고 해도, 여전히 보수정당을 지지하는 시민도 많습니다., title: [전국]전재수 vs 박형준 '초접전'...부산시장 민심 어디로? | YTN, link: https://www.ytn.co.kr/_ln/0115_2026052

In [41]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.tools import tool

@tool
def get_web_search(query: str, search_period: str="w", region: str="kr-kr") -> str:
    """
    특정 지역과 기간을 설정하여 웹 검색을 수행합니다.
    사용자의 질문에 정확하게 답변할 수 있도록 query, search_period, region을 생성하여 도구를 호출합니다.

    Args:
        query: 검색어
        search_period: 검색 기간 (예: {'일간': 'd', '주간': 'w', '월간': 'm', '연간': 'y' 등})
        region: 검색 지역 코드 (예: {'한국': 'kr-kr', '미국': 'us-en', '일본': 'jp-jp' 등})
    
    Returns:
        검색된 결과 문자열
    """
    wrapper = DuckDuckGoSearchAPIWrapper(
        region=region,
        time=search_period
    )

    search = DuckDuckGoSearchResults(
        api_wrapper=wrapper,
        results_separator=";\n"
    )

    searched = search.invoke(query)
    return searched
    

In [42]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools=[get_web_search],
    system_prompt="너는 사용자의 질문에 답변하기 위해 도구를 사용할 수 있다."
)

In [43]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [44]:
response

{'messages': [HumanMessage(content='현재 부산시 지방선거에서 부산시장 후보는 총 몇명이고, 후보의 이름은 누구야?', additional_kwargs={}, response_metadata={}, id='374f25c1-187f-46bc-bbde-96737daadad2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_web_search', 'arguments': '{"query": "\\ud604\\uc7ac \\ubd80\\uc0b0\\uc2dc\\uc7a5 \\ud6c4\\ubcf4"}'}, '__gemini_function_call_thought_signatures__': {'18a1a103-2d2c-413a-b059-8228ce3147ef': 'EjQKMgEMOdbHrvRrdFw7gxFHaH50mmYITrMN1+sFzTcNxPPd5MY5Mxz6q5NdK9l8YMsav6zW'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e77a4-3f6f-7823-b829-a7552154b57e-0', tool_calls=[{'name': 'get_web_search', 'args': {'query': '현재 부산시장 후보'}, 'id': '18a1a103-2d2c-413a-b059-8228ce3147ef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 21, 'total_tokens': 255, 'input_token_details': {'cache_read': 0}}),
  ToolMe